# 4. Daugiasluoksnis perceptronas (MLP)

**Failas:** `winequality/mlp.py` → `mlp_pipeline`  
**Mokymas:** `winequality/tune.py` → `tune_mlp`

Kursų LD2 / MLP tema. Universalus aproksimatorius, bet lentelinėms, asimetriškoms
imtims (chloridai, cukrus, SO₂) dažnai nusileidžia medžiams ar branduolio metodams.


## Formulė (naudojimo kelias, 1 paslėptas sluoksnis)

\[
\mathbf{h} = \mathrm{act}(W_1 \mathbf{z} + \mathbf{b}_1),\qquad
f(\mathbf{z}) = \mathbf{w}_2^\top \mathbf{h} + b_2
\]

\(\mathrm{act} \in \{\mathrm{ReLU},\tanh\}\). Mokymas — Adam su L2 bauda \(\alpha\|W\|^2\)
ir ankstyvuoju stabdymu; mokymo formules egzaminas leidžia praleisti, nes tinklas mokomas vieną kartą.

Hiperparametrai (sluoksniai 8–64, ReLU/tanh, \(\alpha\), mokymosi žingsnis) parenkami
**GroupKFold mokymo bloke**, ne testinėje aibėje.


In [1]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
root = here if (here / "winequality").exists() else here.parent
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "notebooks"))

from nb_pagalba import paruošti, metrikos_lentele, artefaktas
from winequality.config import FEATURE_NAMES, SPLIT_SEED

KIND = "red"  # pakeiskite į "white", jei norite balto vyno
duom = paruošti(KIND)
X_tr, y_tr, X_te, y_te = duom["X_tr"], duom["y_tr"], duom["X_te"], duom["y_te"]
print(f"{KIND}: mokymo n={duom['n_train']}, testo n={duom['n_test']}, sėkla={SPLIT_SEED}")
print("Požymiai:", FEATURE_NAMES)


red: mokymo n=1279, testo n=320, sėkla=0
Požymiai: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


## Artefaktas arba demonstracinis MLP


In [2]:
import joblib
from winequality.mlp import mlp_pipeline

kelias = artefaktas("mlp", KIND)
if kelias.exists():
    vamzdis = joblib.load(kelias)
    print("Įkeltas artefaktas:", kelias.name)
    mlp = vamzdis.named_steps["mlp"]
    print("paslėpti sluoksniai:", mlp.hidden_layer_sizes)
    print("aktyvacija:", mlp.activation, "  alpha:", mlp.alpha)
else:
    print("Artefakto nėra — mokomas mažas MLP.")
    vamzdis = mlp_pipeline(hidden_layer_sizes=(32,), activation="relu", seed=0)
    vamzdis.fit(X_tr, y_tr)
    mlp = vamzdis.named_steps["mlp"]

f = vamzdis.predict(X_te)
print("n_iter_ =", getattr(mlp, "n_iter_", "?"))
print("sluoksnių svorių formos:", [w.shape for w in mlp.coefs_])


Įkeltas artefaktas: mlp_red.joblib
paslėpti sluoksniai: (64,)
aktyvacija: tanh   alpha: 0.0034814103519174673
n_iter_ = 110
sluoksnių svorių formos: [(11, 64), (64, 1)]


## Rankinis tiesioginis sklidimas (1 paslėptas sluoksnis)


In [3]:
def relu(u):
    return np.maximum(u, 0.0)

def tanh_act(u):
    return np.tanh(u)

z = vamzdis.named_steps["prep"].transform(X_te)
W_list, b_list = mlp.coefs_, mlp.intercepts_
h = z
for i, (W, b) in enumerate(zip(W_list[:-1], b_list[:-1])):
    a = h @ W + b
    h = relu(a) if mlp.activation == "relu" else tanh_act(a)
f_rankinis = h @ W_list[-1] + b_list[-1]
f_rankinis = np.asarray(f_rankinis).ravel()
print("max |rankinis forward − sklearn| =", float(np.max(np.abs(f_rankinis - f))))

lentele = metrikos_lentele(y_te, f, "MLP")
display(lentele.round(4))


max |rankinis forward − sklearn| = 0.0


,modelis,MAE,MAE (apvalinta),Acc_0.5,Acc_1.0,κ_w,makro-F1
0,MLP,0.4721,0.3688,0.65,0.8812,0.6182,0.5355


## Ką stebėti

- Raudonam vynui MLP šiame skaidyme beveik lygus SVR (MAE ≈ 0,472).
- Baltam vynui MLP krenta beveik iki tiesinės regresijos (≈ 0,572) — maža lentelinė imtis + asimetriški požymiai.
- Ankstyvasis stabdymas naudoja **10 % mokymo** kaip validaciją, ne užšaldytą testą.
